# Step 5 — Drift Detection: Evidently + River

**Thesis:** Drift-Aware Selective Updating of Two-Stage Tabular ML Pipelines  
**Goal:** Integrate two complementary drift detection approaches:

1. **Evidently (batch)** — computes per-feature KS test / PSI between reference and post-drift windows; exports JSON.
2. **River ADWIN (online)** — streams prediction errors one by one and flags the index where drift is detected.

Then we compare the two detectors: which fires first, and do they agree?

Reference implementations:
- `drift_framework/monitoring/evidently_monitor.py`
- `drift_framework/monitoring/river_monitor.py`

## 5.1 Setup

In [ ]:
import sys, os, json
from pathlib import Path
from typing import Optional, Callable

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from drift_framework.data.loader import load_dataset
from drift_framework.pipeline.two_stage import TwoStagePipeline
from drift_framework.drift.injector import DriftInjector

SEED = 42
RESULTS_DIR = Path(PROJECT_ROOT) / "results"
RESULTS_DIR.mkdir(exist_ok=True)

bundle = load_dataset("adult")

# Fit baseline pipeline
pipe = TwoStagePipeline(
    model_type="xgboost",
    num_features=bundle.num_features,
    cat_features=bundle.cat_features,
)
pipe.fit(bundle.X_ref, bundle.y_ref)
print("Pipeline fitted.")

In [ ]:
# Inject high covariate shift into the post-drift split
injector = DriftInjector(drift_type="covariate", severity="high")
injector.fit(bundle.X_ref, bundle.num_features)
X_drifted, y_drifted = injector.inject(bundle.X_post, bundle.y_post)
print(f"Drifted X shape: {X_drifted.shape}")

## 5.2 Evidently — batch drift detection

Evidently's `DataDriftPreset` applies KS test for numeric features (or chi-squared for categoricals) between the reference and current datasets.

In [ ]:
from evidently.legacy.pipeline.column_mapping import ColumnMapping
from evidently.legacy.report import Report
from evidently.legacy.metric_preset import DataDriftPreset


class EvidentlyMonitor:
    """Batch drift detector using Evidently's DataDriftPreset (KS test by default)."""

    def __init__(self, num_features: list, cat_features: list, stattest: Optional[str] = "ks"):
        self.num_features = num_features
        self.cat_features = cat_features
        self.stattest = stattest
        self._column_mapping = ColumnMapping(
            target=None,
            numerical_features=num_features,
            categorical_features=cat_features,
        )

    def detect(
        self,
        X_reference: pd.DataFrame,
        X_current: pd.DataFrame,
        export_path: Optional[str] = None,
    ) -> dict:
        """Run drift detection; return dict with per-feature statistics."""
        preset_kwargs = {}
        if self.stattest is not None:
            preset_kwargs["stattest"] = self.stattest

        report = Report(metrics=[DataDriftPreset(**preset_kwargs)])
        report.run(
            reference_data=X_reference,
            current_data=X_current,
            column_mapping=self._column_mapping,
        )
        raw = report.as_dict()

        # metrics[0] = DatasetDriftMetric  (dataset-level summary)
        # metrics[1] = DataDriftTable      (per-column breakdown)
        summary_result = raw["metrics"][0]["result"]
        table_result = raw["metrics"][1]["result"] if len(raw["metrics"]) > 1 else summary_result

        dataset_drift = summary_result.get("dataset_drift", False)
        n_drifted = summary_result.get("number_of_drifted_columns", 0)
        drift_by_col = table_result.get("drift_by_columns", {})

        drift_by_feature = {}
        for col_name, col_data in drift_by_col.items():
            drift_by_feature[col_name] = {
                "drift_detected": col_data.get("drift_detected", False),
                "stattest": col_data.get("stattest_name", self.stattest),
                "stat_value": col_data.get("drift_score", None),
                "p_value": col_data.get("p_value", None),
            }

        result = {
            "dataset_drift": dataset_drift,
            "n_drifted_features": n_drifted,
            "drift_by_feature": drift_by_feature,
        }

        if export_path is not None:
            Path(export_path).parent.mkdir(parents=True, exist_ok=True)
            with open(export_path, "w") as f:
                json.dump(result, f, indent=2, default=str)
            print(f"[Evidently] Report saved to {export_path}")

        drifted = [k for k, v in drift_by_feature.items() if v["drift_detected"]]
        print(
            f"[Evidently] Dataset drift: {dataset_drift} | "
            f"Drifted features: {n_drifted} | "
            f"Examples: {drifted[:5]}"
        )
        return result

In [ ]:
ev_monitor = EvidentlyMonitor(
    num_features=bundle.num_features,
    cat_features=bundle.cat_features,
)
ev_result = ev_monitor.detect(
    X_reference=bundle.X_ref,
    X_current=X_drifted,
    export_path=str(RESULTS_DIR / "nb_evidently_adult_covariate_high.json"),
)

In [ ]:
# Display per-feature drift statistics (top 10 by stat_value)
feature_rows = []
for feat, info in ev_result["drift_by_feature"].items():
    feature_rows.append({
        "feature": feat,
        "drift_detected": info["drift_detected"],
        "stat_value": info["stat_value"],
        "p_value": info["p_value"],
    })

feat_df = pd.DataFrame(feature_rows).sort_values("stat_value", ascending=False)
print(f"\nTotal features: {len(feat_df)}  |  Drifted: {ev_result['n_drifted_features']}")
feat_df.head(10)

In [ ]:
# Bar chart: KS statistic per feature
top_features = feat_df.dropna(subset=["stat_value"]).head(15)

fig, ax = plt.subplots(figsize=(12, 4))
colors = ["tomato" if d else "steelblue" for d in top_features["drift_detected"]]
ax.barh(top_features["feature"], top_features["stat_value"], color=colors)
ax.set_xlabel("KS statistic (higher = more drift)")
ax.set_title("Evidently — KS statistic per feature (red = drift detected)")
plt.tight_layout()
plt.show()

## 5.3 Evidently — no-drift baseline (false positive check)

In [ ]:
# Use pre-drift split (same distribution as reference) as current data
ev_nodrift = ev_monitor.detect(
    X_reference=bundle.X_ref,
    X_current=bundle.X_pre,
)
print(f"No-drift scenario — Dataset drift flag: {ev_nodrift['dataset_drift']}")
print(f"Drifted features: {ev_nodrift['n_drifted_features']}")

## 5.4 River ADWIN — online streaming drift detection

ADWIN (Adaptive Windowing) maintains a shrinking window and detects when the mean error rate shifts between sub-windows.

In [ ]:
from river import drift, stream


class RiverMonitor:
    """Online drift detector using ADWIN on the model's prediction error stream."""

    def __init__(self, delta: float = 0.002):
        self.delta = delta

    def detect(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        predict_fn: Callable[[pd.DataFrame], np.ndarray],
    ) -> dict:
        """Stream (X, y) through the model; detect error-rate drift with ADWIN."""
        adwin = drift.ADWIN(delta=self.delta)
        drift_indices = []

        y_arr = y.values
        y_pred_all = predict_fn(X)  # batch predict upfront

        for i, (_x_dict, _y_val) in enumerate(stream.iter_pandas(X, y)):
            error = int(y_pred_all[i] != y_arr[i])
            adwin.update(error)
            if adwin.drift_detected:
                drift_indices.append(i)

        result = {
            "drift_indices": drift_indices,
            "n_drift_detections": len(drift_indices),
            "first_drift_index": drift_indices[0] if drift_indices else None,
        }
        print(
            f"[ADWIN] Total detections: {len(drift_indices)} | "
            f"First at index: {result['first_drift_index']}"
        )
        return result

In [ ]:
river_monitor = RiverMonitor(delta=0.002)
river_result = river_monitor.detect(
    X=X_drifted,
    y=y_drifted,
    predict_fn=pipe.predict,
)

In [ ]:
# Visualize cumulative error rate with ADWIN drift flags
y_pred = pipe.predict(X_drifted)
errors = (y_pred != y_drifted.values).astype(int)
cumulative_error = np.cumsum(errors) / (np.arange(len(errors)) + 1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(cumulative_error, label="Cumulative error rate", color="steelblue")
for idx in river_result["drift_indices"][:10]:  # mark first 10 detections
    ax.axvline(idx, color="tomato", alpha=0.5, linewidth=1)
ax.set_xlabel("Stream index")
ax.set_ylabel("Cumulative error rate")
ax.set_title("River ADWIN — cumulative error rate (red lines = drift detected)")
ax.legend()
plt.tight_layout()
plt.show()

## 5.5 River ADWIN — no-drift baseline

In [ ]:
# Use the pre-drift split (no injected drift)
river_nodrift = river_monitor.detect(
    X=bundle.X_pre,
    y=bundle.y_pre,
    predict_fn=pipe.predict,
)
print(f"No-drift scenario — ADWIN detections: {river_nodrift['n_drift_detections']}")

## 5.6 Detector comparison: Evidently vs River

In [ ]:
print("=== Detector Comparison (high covariate shift) ===")
print(f"Evidently — dataset_drift: {ev_result['dataset_drift']}")
print(f"Evidently — drifted features: {ev_result['n_drifted_features']} / {len(ev_result['drift_by_feature'])}")
print(f"River ADWIN — detections: {river_result['n_drift_detections']}")
print(f"River ADWIN — first detection at index: {river_result['first_drift_index']}")
print()
print("Note: Evidently runs on the full batch upfront; ADWIN flags in real-time.")
print("      ADWIN's first detection index tells us how many rows were needed to detect the shift.")

## 5.7 Sanity checks

In [ ]:
# 1. Evidently detects drift under high covariate shift
assert ev_result["dataset_drift"] is True, \
    "Expected Evidently to detect drift under high covariate shift"
print("Evidently detects drift under high covariate shift — OK")

# 2. Evidently's drifted feature count > 0
assert ev_result["n_drifted_features"] > 0, \
    "Expected at least one drifted feature"
print(f"Evidently drifted features: {ev_result['n_drifted_features']} — OK")

# 3. River ADWIN fires at least once under high drift
assert river_result["n_drift_detections"] > 0, \
    "Expected ADWIN to fire at least once under high covariate shift"
print(f"ADWIN fired {river_result['n_drift_detections']} times — OK")

# 4. Exported JSON exists and is valid
json_path = RESULTS_DIR / "nb_evidently_adult_covariate_high.json"
assert json_path.exists(), "Evidently JSON export not found"
with open(json_path) as f:
    loaded = json.load(f)
assert "dataset_drift" in loaded, "JSON missing 'dataset_drift' key"
print("Evidently JSON export valid — OK")

# 5. Compare Evidently results with framework monitor
from drift_framework.monitoring.evidently_monitor import EvidentlyMonitor as FWEv
fw_ev = FWEv(num_features=bundle.num_features, cat_features=bundle.cat_features)
fw_result = fw_ev.detect(bundle.X_ref, X_drifted)
assert fw_result["dataset_drift"] == ev_result["dataset_drift"], \
    "Notebook Evidently result differs from framework"
print("Notebook Evidently matches framework Evidently — OK")

print("\nAll sanity checks passed!")